In [1]:
%run StdPackages.ipynb
d['processedData'] = os.path.join(d['data'],'processedData') # update to raw data folder

No clean-up of work-folder


# Create $\sigma$ with standard values from GreenREFORM database

Collect in:

In [2]:
db = GpyDB(name = 'GR_EOS')

Load data:

In [3]:
wb = DbFromExcel.simpleLoad(os.path.join(d['data'], 'GR_EOS.xlsx'))
wbDict = {k: pd.DataFrame(wb[k].values) for k in DbFromExcel.sheetnames_from_wb(wb)}

## 1. Production elasticities

For specific elasticities, extract level with industry index:

In [4]:
eDict = {k: pd.Series(wbDict[k].values[1:,1], index = pd.Index(wbDict[k].values[1:, 0], name = 's'), name = k) for k in ('eKELBM','eKELB','eKEL','eKETM','eMKE','eTKE','eREP','eRxE')}

Add export elasticity. This is defined over an additional set $xOth, xTur$ - we simply take the mean over this (we do not have this distinction in the model): 

In [5]:
eDict['eXF'] = pd.Series(wbDict['eXF'].values[1:,2], index = pd.MultiIndex.from_arrays(wbDict['eXF'].values[1:,0:2].T, names = ['x','s']), name = 'eXF').groupby('s').mean()

Collect in one symbol:

In [6]:
σp = pd.concat([v for v in eDict.values()], axis = 1)
σp = σp.fillna(σp.mean(axis = 0)) # Add default values in place of nan
σp = σp.stack().rename_axis(['s','n']) 

C:\Users\sxj477\AppData\Local\Temp\ipykernel_21964\296686641.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  σp = σp.fillna(σp.mean(axis = 0)) # Add default values in place of nan


Add elasticity of demand between foreign/domestic in RxE materials:

In [7]:
eRxE_s = pd.Series(wbDict['eRxE_s'].values[1:,2], index = pd.MultiIndex.from_arrays(wbDict['eRxE_s'].values[1:,0:2].T, names = ['s','n']), name = 'eRxE_s')

Replace zeros with averages (they are likely not used due to sparse patterns in demand, but different RAS adjustments may result in them being used in a different model setting):

In [8]:
eRxE_savg = eRxE_s[eRxE_s>0].groupby('s').mean()
eRxE_s = pd.concat([eRxE_s[eRxE_s==0].add(eRxE_savg), eRxE_s[eRxE_s>0]], axis =0)

Add to symbol:

In [9]:
σp = σp.combine_first(eRxE_s)

Map to new "standard" industry definitions:

In [10]:
mNew = pd.MultiIndex.from_tuples([('10011','10011'), ('10012','10011'), ('10013','10011'), ('35011','Energy')], names = ['s','ss'])
sp = σp.index.levels[0]
mFull = mNew.union(adj.rc_pd(pd.MultiIndex.from_arrays([sp, sp.rename('ss')]), ('not', mNew.droplevel('ss'))))
σp = adjMultiIndex.applyMult(σp, mFull).groupby(['ss','n']).mean().rename_axis(index = {'ss':'s'})

Do this as well for the $n$ index:

In [11]:
mNew = mNew.rename(['n','nn'])
σp_new = adjMultiIndex.applyMult(adj.rc_pd(σp, mNew), mNew).groupby(['s','nn']).mean().rename_axis(index={'nn':'n'})
σp = σp_new.combine_first(adj.rc_pd(σp, ('not', mNew)))

Add to database:

In [12]:
db['σp'] = σp

## X. Export

In [13]:
db.export(repo = d['processedData'])